# 00 · Offline Quickstart (`contoso-smoke.yaml`)

Run an end-to-end `cfeval` evaluation in **under 10 seconds with zero cloud credentials**:
1. Load the 8-question v2 smoke suite (`data/golden/contoso-smoke.yaml`).
2. Run `EvalHarness` in offline mock mode comparing an `untuned` baseline against a `tuned` semantic model.
3. Compute `summarize_run_stats` (Wilson 95% CI, `pass@k`, `pass^k`) and `compare_runs` (paired McNemar `p`-value).
4. Render a self-contained HTML scorecard.

In [ ]:
import html
import math
import pathlib
import sys
import yaml

repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

import cfeval
from cfeval.cli import build_harness
from cfeval.config import Settings
from cfeval.golden import GoldenQuestion, GoldenSetError, load_golden

MOCK = True
smoke_path = repo / "data" / "golden" / "contoso-smoke.yaml"

try:
    questions = load_golden(smoke_path)
except GoldenSetError:
    doc = yaml.safe_load(smoke_path.read_text(encoding="utf-8"))
    defaults = doc.get("defaults", {})
    decoys_default = ";".join(defaults.get("decoys", defaults.get("decoy_models", [])))
    questions = [
        GoldenQuestion(
            id=item["id"],
            question=item["question"],
            category=item.get("category") or (item.get("tags") or ["general"])[0],
            expected_artifact=item.get("expected_artifact") or "",
            artifact_aliases=";".join(item.get("artifact_aliases", [])),
            expected_model=item.get("expected_model") or "",
            decoys=";".join(item.get("decoys", item.get("decoy_models", []))) or decoys_default,
            expected_value=float(item["expected"]) if item.get("expected") is not None else None,
            unit=item.get("unit", "number") if item.get("unit") in ("currency", "percent", "count", "number") else "number",
            rel_tol=float(item.get("rel_tol", item.get("tolerance", defaults.get("rel_tol", 0.005)))),
            abs_tol=float(item.get("abs_tol", 0.0)),
            reference_dax=item.get("reference_dax", item.get("grounding_query_dax", "")),
            criticality=item.get("criticality") or ("critical" if item.get("critical") else "standard"),
        )
        for item in doc["questions"]
    ]

print(f"cfeval {cfeval.__version__} | Loaded {len(questions)} smoke cases from {smoke_path.name} (MOCK={MOCK})")

In [ ]:
out_root = repo / "build" / "notebook-runs"
settings = Settings.load(env_file=None, output_dir=str(out_root))

try:
    untuned_run = build_harness(
        settings, mock=MOCK, publish=False, profile="untuned", seed=7,
        questions=questions, run_id="nb00-untuned", run_name="Untuned baseline",
    ).run()
except KeyError:
    untuned_run = build_harness(
        settings, mock=MOCK, publish=False, profile="baseline", seed=7,
        questions=questions, run_id="nb00-untuned", run_name="Untuned baseline",
    ).run()

tuned_run = build_harness(
    settings, mock=MOCK, publish=False, profile="tuned", seed=7,
    questions=questions, run_id="nb00-tuned", run_name="Tuned semantic model",
).run()

print("Untuned pass rate:", untuned_run.metrics["PassRate"])
print("Tuned pass rate:  ", tuned_run.metrics["PassRate"])

In [ ]:
try:
    from cfeval.stats import compare_runs, summarize_run_stats  # type: ignore[import-not-found]
except ImportError:
    def _wilson_ci(successes: int, total: int, z: float = 1.959964) -> tuple[float, float]:
        if total <= 0:
            return (0.0, 0.0)
        p = successes / total
        denom = 1.0 + (z * z) / total
        center = (p + (z * z) / (2.0 * total)) / denom
        half = (z * math.sqrt((p * (1.0 - p) + (z * z) / (4.0 * total)) / total)) / denom
        return (round(max(0.0, center - half), 4), round(min(1.0, center + half), 4))

    def summarize_run_stats(results: list[dict]) -> dict:
        by_q: dict[str, list[bool]] = {}
        for row in results:
            by_q.setdefault(row["QuestionId"], []).append(row["Verdict"] == "PASS")
        n_q = len(by_q)
        pass_at_k = sum(1 for flags in by_q.values() if any(flags)) / max(n_q, 1)
        pass_pow_k = sum(1 for flags in by_q.values() if all(flags)) / max(n_q, 1)
        passed = sum(1 for row in results if row["Verdict"] == "PASS")
        ci_lo, ci_hi = _wilson_ci(passed, len(results))
        return {
            "questions": n_q,
            "pass_rate": round(passed / max(len(results), 1), 4),
            "wilson_95_ci": (ci_lo, ci_hi),
            "pass_at_k": round(pass_at_k, 4),
            "pass_pow_k": round(pass_pow_k, 4),
        }

    def compare_runs(baseline_results: list[dict], candidate_results: list[dict]) -> dict:
        base_map = {r["QuestionId"]: r["Verdict"] == "PASS" for r in baseline_results}
        cand_map = {r["QuestionId"]: r["Verdict"] == "PASS" for r in candidate_results}
        shared = sorted(set(base_map) & set(cand_map))
        regressions = [qid for qid in shared if base_map[qid] and not cand_map[qid]]
        improvements = [qid for qid in shared if not base_map[qid] and cand_map[qid]]
        b, c = len(regressions), len(improvements)
        n_disc = b + c
        if n_disc == 0:
            p_val = 1.0
        else:
            k_min = min(b, c)
            tail = sum(math.comb(n_disc, i) for i in range(k_min + 1)) / (2 ** n_disc)
            p_val = round(min(1.0, 2.0 * tail), 6)
        return {
            "shared_questions": len(shared),
            "improvements": improvements,
            "regressions": regressions,
            "mcnemar_p_value": p_val,
        }

untuned_stats = summarize_run_stats(untuned_run.results)
tuned_stats = summarize_run_stats(tuned_run.results)
diff = compare_runs(untuned_run.results, tuned_run.results)
print("Untuned stats:", untuned_stats)
print("Tuned stats:  ", tuned_stats)
print("Paired diff:  ", diff)

In [ ]:
report_path = tuned_run.folder / "report.html"
try:
    from cfeval.report import render_html_report  # type: ignore[import-not-found]
    html_doc = render_html_report(tuned_run.folder)
    if isinstance(html_doc, pathlib.Path):
        report_path = html_doc
    else:
        report_path.write_text(str(html_doc), encoding="utf-8")
except ImportError:
    rows_html = "\n".join(
        f"<tr><td>{html.escape(r['QuestionId'])}</td><td>{html.escape(r['Verdict'])}</td>"
        f"<td>{r['ValueScore']}</td><td>{html.escape(r.get('ExpectedModel') or '')}</td>"
        f"<td>{html.escape(r.get('QueriedModels') or '')}</td></tr>"
        for r in tuned_run.results
    )
    report_path.write_text(
        "<!doctype html><html><head><meta charset='utf-8'><title>cfeval Scorecard</title></head><body>"
        f"<h1>cfeval Offline Quickstart — {html.escape(tuned_run.run_id)}</h1>"
        f"<p>Pass rate: {tuned_run.metrics['PassRate']:.1%}</p>"
        "<table border='1' cellpadding='6'><tr><th>ID</th><th>Verdict</th><th>Value</th><th>Expected Model</th><th>Queried Model</th></tr>"
        f"{rows_html}</table></body></html>",
        encoding="utf-8",
    )

print(f"Rendered HTML scorecard ({report_path.stat().st_size} bytes): {report_path}")